# LLM-Based Action Planning for Network Intrusion Detection

This notebook uses saved prediction results, SHAP explanations, and RAG documents to generate LLM-based action plans for detected attacks.

In [ ]:
# 1. Setup and Imports
# -----------------------------
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Check required files
required_files = {
    "predictions": "outputs/sample_predictions_detailed.json",
    "shap_explanations": "outputs/sample_shap_explanations.csv",
    "mitigation_summary": "outputs/vfl_shap_mitigation_summary.txt",
    "agent_mitigation": "outputs/vfl_shap_agent_mitigation_summary.csv",
    "global_summary": "outputs/vfl_shap_global_summary.csv"
}

print("Checking required files...")
for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗'} {path}")
    if not exists:
        print(f"    WARNING: {path} not found!")

# Create output directory for action plans
os.makedirs("outputs/action_plans", exist_ok=True)
print("\nOutput directory created: outputs/action_plans")
# -----------------------------

In [ ]:
# 2. Load Saved Results and RAG Documents
# -----------------------------

# Load prediction results
with open(required_files["predictions"], "r", encoding="utf-8") as f:
    predictions_data = json.load(f)

# Load SHAP explanations
shap_df = pd.read_csv(required_files["shap_explanations"])

# Load metadata
with open("outputs/sample_predictions_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Load RAG documents
rag_docs = {}

# Load mitigation summary text
if os.path.exists(required_files["mitigation_summary"]):
    with open(required_files["mitigation_summary"], "r", encoding="utf-8") as f:
        rag_docs["mitigation_summary"] = f.read()

# Load agent mitigation CSV
if os.path.exists(required_files["agent_mitigation"]):
    rag_docs["agent_mitigation_df"] = pd.read_csv(required_files["agent_mitigation"])

# Load global summary
if os.path.exists(required_files["global_summary"]):
    rag_docs["global_summary_df"] = pd.read_csv(required_files["global_summary"])

print(f"Loaded {len(predictions_data)} prediction results")
print(f"Loaded {len(shap_df)} SHAP explanations")
print(f"Loaded RAG documents: {list(rag_docs.keys())}")
print(f"\nParty names: {metadata.get('party_names', [])}")
print(f"Label mapping: {metadata.get('label_mapping', {})}")
# -----------------------------

In [ ]:
# 3. RAG Knowledge Retrieval Functions
# -----------------------------

def retrieve_mitigation_recommendations(attack_type, rag_docs):
    """
    Retrieve mitigation recommendations for a specific attack type from RAG documents.
    """
    recommendations = {
        "text_recommendations": "",
        "party_actions": [],
        "global_stats": []
    }
    
    # Extract from text file
    if "mitigation_summary" in rag_docs:
        content = rag_docs["mitigation_summary"]
        attack_upper = attack_type.upper()
        
        # Find section for this attack type
        start_marker = f"{attack_upper} Attack - Mitigation Recommendations"
        start_idx = content.find(start_marker)
        
        if start_idx != -1:
            # Find end of section (next attack or end of file)
            end_idx = content.find("\n\n", start_idx + 200)
            if end_idx == -1:
                end_idx = len(content)
            
            recommendations["text_recommendations"] = content[start_idx:end_idx]
    
    # Extract from agent mitigation CSV
    if "agent_mitigation_df" in rag_docs:
        df = rag_docs["agent_mitigation_df"]
        attack_rows = df[df['Class'].str.upper() == attack_type.upper()]
        
        if len(attack_rows) > 0:
            for _, row in attack_rows.iterrows():
                recommendations["party_actions"].append({
                    "party": row['Party'],
                    "domain": row['Domain'],
                    "contribution": row['Mean_contrib'],
                    "suggested_action": row['Suggested_Action'],
                    "is_dominant": row['Is_Dominant']
                })
    
    # Get global stats
    if "global_summary_df" in rag_docs:
        recommendations["global_stats"] = rag_docs["global_summary_df"].to_dict('records')
    
    return recommendations

def get_party_info(party_name, rag_docs):
    """
    Get detailed information about a specific party from RAG documents.
    """
    party_info = {
        "name": party_name,
        "domain": "",
        "global_contribution": 0.0
    }
    
    if "global_summary_df" in rag_docs:
        df = rag_docs["global_summary_df"]
        party_row = df[df['Party'] == party_name]
        if len(party_row) > 0:
            row = party_row.iloc[0]
            party_info["domain"] = row.get('Domain', '')
            party_info["global_contribution"] = row.get('Mean_contrib_All', 0.0)
    
    return party_info

print("RAG retrieval functions defined.")
# -----------------------------

In [ ]:
# 4. LLM Prompt Creation
# -----------------------------

def create_llm_prompt(prediction_result, shap_explanation, rag_knowledge, party_info):
    """
    Create a comprehensive prompt for LLM reasoning based on prediction, SHAP, and RAG.
    """
    attack_type = prediction_result.get('predicted_label', 'UNKNOWN')
    confidence = prediction_result.get('confidence', 0.0)
    
    # Format party contributions
    party_contribs = shap_explanation.get('party_contributions_pct', {})
    party_names_list = list(party_contribs.keys())
    party_values_list = list(party_contribs.values())
    
    prompt = f"""
You are a cybersecurity AI assistant analyzing a network intrusion detection result.

## DETECTION RESULTS
Attack Type Detected: {attack_type}
Confidence Score: {confidence:.2%}
Prediction Correct: {prediction_result.get('is_correct', False)}

All Class Probabilities:
{json.dumps(prediction_result.get('all_probabilities', {}), indent=2)}

## SHAP EXPLANATION (Which Sensor Detected This Attack)
Dominant Sensor/Party: {shap_explanation.get('dominant_party', 'Unknown')}
Dominant Contribution: {shap_explanation.get('dominant_contribution_pct', 0):.2%}

Party Contributions:
"""
    
    for party_name, contrib_pct in party_contribs.items():
        prompt += f"  - {party_name}: {contrib_pct:.2%}\n"
    
    prompt += f"""

## DOMINANT PARTY INFORMATION
Party Name: {party_info.get('name', 'Unknown')}
Domain: {party_info.get('domain', 'Unknown')}
Global Contribution: {party_info.get('global_contribution', 0):.2%}

## RETRIEVED KNOWLEDGE BASE (RAG)
### Mitigation Recommendations:
{rag_knowledge.get('text_recommendations', 'Not available')[:1000]}...

### Party-Specific Actions:
"""
    
    for party_action in rag_knowledge.get('party_actions', [])[:5]:  # Limit to 5 for prompt size
        prompt += f"""
Party: {party_action.get('party', 'Unknown')}
  Domain: {party_action.get('domain', 'Unknown')}
  Contribution: {party_action.get('contribution', 0):.2%}
  Suggested Action: {party_action.get('suggested_action', 'None')}
  Is Dominant: {party_action.get('is_dominant', False)}
"""
    
    prompt += """

## TASK
Based on the detection results, SHAP explanation, and retrieved knowledge base:

1. **Threat Assessment**: Analyze the threat level (Low/Medium/High/Critical)
   - Consider: confidence score, attack type severity, SHAP contribution strength

2. **Action Plan**: Recommend specific mitigation actions
   - Primary actions from the dominant sensor/party (based on RAG recommendations)
   - Supporting actions from other parties
   - Consider the retrieved knowledge base recommendations

3. **Reasoning**: Explain why these actions are appropriate
   - Why the dominant party detected this attack
   - How the actions address the specific attack type
   - Alignment with security best practices from RAG

4. **Framework Alignment**: Reference relevant security frameworks
   - NIST Cybersecurity Framework controls (if applicable)
   - OWASP Top 10 recommendations (if applicable)

5. **Execution Priority**: Determine action execution priority
   - Immediate (Critical threats, high confidence >0.9)
   - Fast-track (High threats, medium-high confidence 0.7-0.9)
   - Standard workflow (Medium threats, confidence 0.5-0.7)
   - Monitor only (Low threats or low confidence <0.5)

## OUTPUT FORMAT
Provide your response in the following JSON format:
{
  "threat_level": "Critical|High|Medium|Low",
  "threat_justification": "Brief explanation of threat level",
  "primary_actions": [
    "Action 1",
    "Action 2",
    ...
  ],
  "supporting_actions": [
    "Action 1",
    "Action 2",
    ...
  ],
  "reasoning": "Detailed explanation of why these actions are appropriate",
  "dominant_party_role": "Explanation of why the dominant party detected this attack",
  "framework_alignment": {
    "nist_controls": ["Control 1", "Control 2"],
    "owasp_recommendations": ["Recommendation 1", "Recommendation 2"]
  },
  "execution_priority": "Immediate|Fast-track|Standard|Monitor",
  "confidence_in_recommendations": "High|Medium|Low"
}

Begin your analysis:
"""
    
    return prompt

print("LLM prompt creation function defined.")
# -----------------------------

In [ ]:
# 5. LLM Integration (Replace with Actual LLM API)
# -----------------------------

def call_llm_api(prompt):
    """
    Call LLM API to get action plan.
    
    TODO: Replace this with actual LLM API call:
    - OpenAI: openai.ChatCompletion.create(...)
    - Anthropic: anthropic.Anthropic().messages.create(...)
    - Local LLM: Use your local LLM API
    """
    
    # Example: OpenAI API call (uncomment and configure)
    # import openai
    # openai.api_key = "your-api-key"
    # 
    # response = openai.ChatCompletion.create(
    #     model="gpt-4",
    #     messages=[
    #         {"role": "system", "content": "You are a cybersecurity expert."},
    #         {"role": "user", "content": prompt}
    #     ],
    #     temperature=0.3
    # )
    # 
    # llm_response_text = response.choices[0].message.content
    # 
    # # Parse JSON from response
    # import re
    # json_match = re.search(r'\{.*\}', llm_response_text, re.DOTALL)
    # if json_match:
    #     return json.loads(json_match.group())
    # else:
    #     return parse_structured_response(llm_response_text)
    
    # For now, return a template (replace with actual LLM call)
    print("="*80)
    print("LLM API CALL (Simulated)")
    print("="*80)
    print(f"Prompt length: {len(prompt)} characters")
    print(f"First 500 chars: {prompt[:500]}...")
    print("\nNOTE: Replace call_llm_api() with actual LLM API integration")
    print("="*80)
    
    # Template response (replace with actual LLM response)
    template_response = {
        "threat_level": "High",
        "threat_justification": "High confidence attack detection with significant SHAP contribution from dominant party",
        "primary_actions": [
            "Execute primary mitigation actions from dominant party based on RAG recommendations",
            "Monitor attack progression and collect additional evidence",
            "Log all related network flows for forensic analysis"
        ],
        "supporting_actions": [
            "Enable enhanced monitoring from other parties",
            "Collect additional evidence from non-dominant sensors",
            "Alert security team with detailed SHAP explanation"
        ],
        "reasoning": "Based on SHAP analysis, the dominant party has strong contribution to detection. The RAG knowledge base provides specific mitigation recommendations for this attack type. Actions should prioritize the capabilities of the dominant party while maintaining support from other sensors.",
        "dominant_party_role": "The dominant party's sensor type is most effective for detecting this attack pattern based on its domain expertise and feature set.",
        "framework_alignment": {
            "nist_controls": ["PR.AC-5", "DE.AE-1", "RS.AN-1", "RS.CO-2"],
            "owasp_recommendations": ["Implement rate limiting", "Enable WAF rules", "Apply security patches"]
        },
        "execution_priority": "Fast-track",
        "confidence_in_recommendations": "High"
    }
    
    return template_response

print("LLM integration function defined (ready for API integration).")
# -----------------------------

In [ ]:
# 6. Process All Samples and Generate Action Plans
# -----------------------------

def process_sample_with_llm(sample_data, rag_docs, metadata):
    """
    Process a single sample through the LLM action planning pipeline.
    """
    # Get prediction result
    prediction_result = {
        "sample_id": sample_data.get("sample_id"),
        "predicted_label": sample_data.get("predicted_label"),
        "true_label": sample_data.get("true_label"),
        "confidence": sample_data.get("confidence"),
        "all_probabilities": sample_data.get("all_probabilities", {}),
        "is_correct": sample_data.get("is_correct", False)
    }
    
    # Get SHAP explanation
    shap_explanation = sample_data.get("shap_explanation", {})
    
    # Retrieve RAG knowledge
    attack_type = prediction_result.get("predicted_label", "UNKNOWN")
    rag_knowledge = retrieve_mitigation_recommendations(attack_type, rag_docs)
    
    # Get party info
    dominant_party = shap_explanation.get("dominant_party", "Unknown")
    party_info = get_party_info(dominant_party, rag_docs)
    
    # Create LLM prompt
    prompt = create_llm_prompt(prediction_result, shap_explanation, rag_knowledge, party_info)
    
    # Call LLM
    llm_response = call_llm_api(prompt)
    
    # Compile complete result
    result = {
        "sample_id": prediction_result["sample_id"],
        "prediction": prediction_result,
        "shap_explanation": shap_explanation,
        "rag_knowledge": {
            "attack_type": attack_type,
            "has_recommendations": len(rag_knowledge.get("text_recommendations", "")) > 0,
            "num_party_actions": len(rag_knowledge.get("party_actions", []))
        },
        "llm_action_plan": llm_response,
        "prompt_used": prompt,
        "generation_timestamp": datetime.now().isoformat()
    }
    
    return result

# Process all samples
print("="*80)
print("PROCESSING SAMPLES AND GENERATING ACTION PLANS")
print("="*80)

action_plans = []
num_samples = min(10, len(predictions_data))  # Process first 10 samples (adjust as needed)

print(f"\nProcessing {num_samples} samples...")
for idx, sample_data in enumerate(predictions_data[:num_samples]):
    print(f"\n[{idx+1}/{num_samples}] Processing sample {sample_data.get('sample_id')}...")
    
    try:
        result = process_sample_with_llm(sample_data, rag_docs, metadata)
        action_plans.append(result)
        
        # Print summary
        attack_type = result['prediction']['predicted_label']
        threat_level = result['llm_action_plan'].get('threat_level', 'Unknown')
        priority = result['llm_action_plan'].get('execution_priority', 'Unknown')
        print(f"  Attack: {attack_type} | Threat: {threat_level} | Priority: {priority}")
        
    except Exception as e:
        print(f"  ERROR processing sample {sample_data.get('sample_id')}: {str(e)}")
        continue

print(f"\n\nProcessed {len(action_plans)} samples successfully.")
# -----------------------------

In [ ]:
# 7. Save Action Plans
# -----------------------------

# Save detailed action plans as JSON
output_file = "outputs/action_plans/llm_action_plans_detailed.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(action_plans, f, indent=2, ensure_ascii=False)

# Create summary DataFrame
summary_data = []
for plan in action_plans:
    summary_data.append({
        "sample_id": plan["sample_id"],
        "predicted_label": plan["prediction"]["predicted_label"],
        "true_label": plan["prediction"]["true_label"],
        "confidence": plan["prediction"]["confidence"],
        "is_correct": plan["prediction"]["is_correct"],
        "dominant_party": plan["shap_explanation"]["dominant_party"],
        "threat_level": plan["llm_action_plan"].get("threat_level", "Unknown"),
        "execution_priority": plan["llm_action_plan"].get("execution_priority", "Unknown"),
        "num_primary_actions": len(plan["llm_action_plan"].get("primary_actions", [])),
        "num_supporting_actions": len(plan["llm_action_plan"].get("supporting_actions", []))
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("outputs/action_plans/llm_action_plans_summary.csv", index=False)

# Save action plans in readable format
readable_output = []
readable_output.append("="*80)
readable_output.append("LLM-BASED ACTION PLANS FOR NETWORK INTRUSION DETECTION")
readable_output.append("="*80)
readable_output.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
readable_output.append(f"Total samples: {len(action_plans)}")
readable_output.append("\n" + "="*80)

for plan in action_plans:
    readable_output.append(f"\n\n{'='*80}")
    readable_output.append(f"SAMPLE {plan['sample_id']}")
    readable_output.append(f"{'='*80}")
    
    pred = plan["prediction"]
    shap = plan["shap_explanation"]
    llm = plan["llm_action_plan"]
    
    readable_output.append(f"\nDetection:")
    readable_output.append(f"  Attack Type: {pred['predicted_label']}")
    readable_output.append(f"  Confidence: {pred['confidence']:.2%}")
    readable_output.append(f"  True Label: {pred['true_label']}")
    readable_output.append(f"  Correct: {pred['is_correct']}")
    
    readable_output.append(f"\nSHAP Explanation:")
    readable_output.append(f"  Dominant Party: {shap['dominant_party']}")
    readable_output.append(f"  Dominant Contribution: {shap['dominant_contribution_pct']:.2%}")
    
    readable_output.append(f"\nLLM Action Plan:")
    readable_output.append(f"  Threat Level: {llm.get('threat_level', 'Unknown')}")
    readable_output.append(f"  Execution Priority: {llm.get('execution_priority', 'Unknown')}")
    readable_output.append(f"\n  Primary Actions:")
    for i, action in enumerate(llm.get('primary_actions', []), 1):
        readable_output.append(f"    {i}. {action}")
    readable_output.append(f"\n  Supporting Actions:")
    for i, action in enumerate(llm.get('supporting_actions', []), 1):
        readable_output.append(f"    {i}. {action}")
    readable_output.append(f"\n  Reasoning:")
    readable_output.append(f"    {llm.get('reasoning', 'N/A')}")
    readable_output.append(f"\n  Framework Alignment:")
    readable_output.append(f"    NIST: {', '.join(llm.get('framework_alignment', {}).get('nist_controls', []))}")
    readable_output.append(f"    OWASP: {', '.join(llm.get('framework_alignment', {}).get('owasp_recommendations', []))}")

readable_content = "\n".join(readable_output)
with open("outputs/action_plans/llm_action_plans_readable.txt", "w", encoding="utf-8") as f:
    f.write(readable_content)

print("="*80)
print("ACTION PLANS SAVED")
print("="*80)
print(f"\nFiles saved:")
print(f"  - outputs/action_plans/llm_action_plans_detailed.json")
print(f"  - outputs/action_plans/llm_action_plans_summary.csv")
print(f"  - outputs/action_plans/llm_action_plans_readable.txt")
print(f"\nTotal action plans generated: {len(action_plans)}")
print(f"\nThreat level distribution:")
print(summary_df['threat_level'].value_counts())
print(f"\nExecution priority distribution:")
print(summary_df['execution_priority'].value_counts())
# -----------------------------